# Análisis Exploratorio de Features - ONU y OECD

Este notebook contiene la exploración inicial de las variables de características de desarrollo urbano procedentes de las bases de datos de ONU-Hábitat y OECD.

El objetivo es comprender la estructura de cada dataset, limpiar sus columnas y evaluar la cobertura y calidad de los datos para la posterior integración.

In [25]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Encontrar la raíz del proyecto para asegurar que las rutas se resuelvan correctamente
def obtener_raiz_proyecto():
    current = os.path.abspath(os.getcwd())
    while current != os.path.dirname(current):
        if '.git' in os.listdir(current) or 'README.md' in os.listdir(current):
            return current
        current = os.path.dirname(current)
    return os.getcwd()

# Función auxiliar para cargar archivos en Windows lidiando con límites de ruta larga (MAX_PATH)
def cargar_excel_seguro(path_relativo, sheet_name=0, skiprows=0):
    raiz = obtener_raiz_proyecto()
    abs_path = os.path.abspath(os.path.join(raiz, path_relativo))
    if os.name == 'nt' and len(abs_path) >= 260:
        abs_path = '\\?\\' + abs_path
    return pd.read_excel(abs_path, sheet_name=sheet_name, skiprows=skiprows)


## 1. City Prosperity Index (CPI)

El CPI mide el desarrollo de las ciudades en múltiples dimensiones (productividad, infraestructura, calidad de vida, equidad, sostenibilidad ambiental y gobernanza).

Este dataset tiene la particularidad de contar con cabeceras de múltiples niveles que aplanaremos para obtener nombres de columnas limpios.

In [26]:
path_cpi = 'data/raw/urban-indicator-database-features/city-prosperity-index-cpi/city_prosperity_index_database_2016.xls'
df_cpi_raw = cargar_excel_seguro(path_cpi, sheet_name='City Prosperity Index', skiprows=0)

# Aplanar cabeceras multinivel propagando hacia la derecha (ffill)
headers = df_cpi_raw.iloc[0:4].ffill(axis=1).fillna('')
new_columns = []
for col_idx in range(df_cpi_raw.shape[1]):
    parts = [str(headers.iloc[row_idx, col_idx]).strip() for row_idx in range(4)]
    clean_parts = []
    for p in parts:
        p_clean = p.replace('.0', '')
        if p_clean and p_clean != 'nan' and p_clean not in clean_parts:
            clean_parts.append(p_clean)
    new_columns.append(' | '.join(clean_parts))

df_cpi = df_cpi_raw.iloc[4:].copy()
df_cpi.columns = new_columns

# Renombrar columnas clave de identificación
df_cpi = df_cpi.rename(columns={
    'City Prosperity Index. Global CPI. Base Data. 2016 | COUNTRY': 'Country',
    'City Prosperity Index. Global CPI. Base Data. 2016 | CITY': 'City',
    'City Prosperity Index. Global CPI. Base Data. 2016 | YEAR': 'Year'
})

# Filtrar filas vacías o de cabecera de continente
df_cpi = df_cpi.dropna(subset=['City'])
df_cpi = df_cpi[df_cpi['City'] != '--']

print(f'Dimensiones de CPI: {df_cpi.shape[0]} ciudades x {df_cpi.shape[1]} columnas')
df_cpi[['Country', 'City', 'City Prosperity Index. Global CPI. Base Data. 2016 | PRODUCTIVITY INDEX (P) | Economic Strenght | City Product per capita']].head(3)

Dimensiones de CPI: 333 ciudades x 35 columnas


,Country,City,City Prosperity Index. Global CPI. Base Data. 2016 | PRODUCTIVITY INDEX (P) | Economic Strenght | City Product per capita
6,Ghana,Accra,3430.650941
7,Kenya,Nairobi,8208.962453
8,Liberia,Monrovia,707.522513


## 2. Coeficiente de Gini (Desigualdad Económica)

El coeficiente de Gini mide la desigualdad de ingresos de las ciudades. Contiene registros anuales del período 2010 - 2017.

In [27]:
path_gini = 'data/raw/urban-indicator-database-features/economic-indicators/Gini_at_disposable_income_after_taxes_and_transfers,_2010_-_2017.xls'
df_gini_raw = cargar_excel_seguro(path_gini, skiprows=0)

# Reconstruir las columnas con el primer registro de fila que contiene los años
years = df_gini_raw.iloc[0].tolist()
df_gini = df_gini_raw.iloc[1:].copy()
df_gini.columns = ['Country', 'City_Region'] + [str(int(float(y))) for y in years[2:]]

print(f'Dimensiones de Gini: {df_gini.shape[0]} registros x {df_gini.shape[1]} columnas')
df_gini.head(3)

Dimensiones de Gini: 227 registros x 10 columnas


,Country,City_Region,2010,2011,2012,2013,2014,2015,2016,2017
1,Kazakhstan,Akmola,0.266,0.284,0.276,0.267,0.277,0.27,0.268,0.273
2,Kazakhstan,Aktobe,0.271,0.278,0.275,0.259,0.261,0.269,0.244,0.245
3,Kazakhstan,Almaty,0.263,0.246,0.254,0.245,0.247,0.255,0.257,0.275


## 3. Espacios Abiertos y Áreas Verdes

Analizamos el porcentaje de área verde per cápita y la participación de espacios públicos de uso abierto (indicador ODS 11.7.1).

In [28]:
path_areas_verdes = 'data/raw/urban-indicator-database-features/open-spaces-and-green-areas/green_areas_1990_2020.xlsx'
df_verdes = cargar_excel_seguro(path_areas_verdes, sheet_name='Data')

# Limpieza básica de columnas de identificación
df_verdes = df_verdes.rename(columns={'Country or Territory Name': 'Country', 'City Name': 'City'})
print(f'Dimensiones de Áreas Verdes: {df_verdes.shape[0]} ciudades x {df_verdes.shape[1]} columnas')
df_verdes.head(3)

Dimensiones de Áreas Verdes: 667 ciudades x 15 columnas


,Country,City Code,City,SDG Sub-Region,SDG Region,Average share of green area in city/urban area 1990 (%),Average share of green area in city/ urban area 2000 (%),Average share of green area in city/ urban area 2010 (%),Average share of green area in city/ urban area 2020 (%),Green area per capita 1990 (m2/person),Green area per capita 2000 (m2/person),Green area per capita 2010 (m2/person),Green area per capita 2020 (m2/person),Data Source,FootNote
0,Afghanistan,AF_KABUL,Kabul,Southern Asia,Central and Southern Asia,5.494391,5.798643,3.895394,3.509685,7.752777,5.271168,2.594089,1.714164,"UN-Habitat Urban Indicators Database, 2553",Data on green areas per city is produced from ...
1,Afghanistan,AF_HERAT,Herat,Southern Asia,Central and Southern Asia,11.016731,5.147899,3.807389,3.224931,25.021003,7.792622,3.587259,1.746983,"UN-Habitat Urban Indicators Database, 2571",Data on green areas per city is produced from ...
2,Afghanistan,AF_MAZAR_E_SHARIF,Mazar-e Sharif,Southern Asia,Central and Southern Asia,2.627921,0.428270,0.723127,1.209696,6.892556,1.002613,1.289664,1.450578,"UN-Habitat Urban Indicators Database, 2655",Data on green areas per city is produced from ...


## 4. Crecimiento Espacial de Ciudades

Este indicador evalúa la tasa de crecimiento espacial (área urbana construida en kilómetros cuadrados) histórica.

In [29]:
path_crecimiento = 'data/raw/urban-indicator-database-features/spatial-growth-of-cities-and-urban-areas/spatial_growth_of_cities_and_urban_areas.xlsx'
df_crecimiento = cargar_excel_seguro(path_crecimiento, sheet_name='Sheet1')

df_crecimiento = df_crecimiento.rename(columns={'Country or Territory Name': 'Country', 'City Name': 'City'})
print(f'Dimensiones de Crecimiento Espacial: {df_crecimiento.shape[0]} ciudades x {df_crecimiento.shape[1]} columnas')
df_crecimiento.head(3)

Dimensiones de Crecimiento Espacial: 2084 ciudades x 23 columnas


,SDG Goal,SDG Target,SDG Indicator,Country or Territory Code,Country,SDG Region,SDG Sub-Region,City Code,City,Data Year 1,...,Land consumption rate (LCR) Year 2 to Year 3 (%),Population Growth Rate (PGR) Year 1 to Year 2 (%),Population Growth Rate (PGR) Year 2 to Year 3 (%),Ratio of Land Consumption Rate to Population Growth Rate (LCRPGR) Year 1 to Year 2 (Ratio),Ratio of Land Consumption Rate to Population Growth Rate (LCRPGR) Year 2 to Year 3 (Ratio),Built Up area Per Capita Year 1 (m2 per person),Built Up area Per Capita Year 2 (m2 per person),Built Up area Per Capita Year 3 (m2 per person),Data Source,FootNote
0,11,11.3,11.3.1,170,Colombia,Latin America and the Caribbean,South America,CO_NAR_MOSQUERA,Nariño_Mosquera,2018,...,NaN,0.032793,NaN,0.506699,NaN,64.252974,59.066949,NaN,National Administrative Department of Statisti...,Designation and data provided by the country a...
1,11,11.3,11.3.1,170,Colombia,Latin America and the Caribbean,South America,CO_CUN_GRANADA,Cundinamarca_Granada,2018,...,NaN,0.027452,NaN,0.092294,NaN,98.628728,87.068042,NaN,National Administrative Department of Statisti...,Designation and data provided by the country a...
2,11,11.3,11.3.1,170,Colombia,Latin America and the Caribbean,South America,CO_ANT_CALDAS,Antioquia_Caldas,2018,...,NaN,0.025756,NaN,0.109575,NaN,30.688711,27.361272,NaN,National Administrative Department of Statisti...,Designation and data provided by the country a...


## 5. Población de Aglomeraciones Urbanas

Población histórica y proyectada (en miles) para aglomeraciones urbanas de más de 300,000 habitantes.

In [30]:
path_poblacion = 'data/raw/urban-indicator-database-features/urban-population-trends/Population_in_Urban_Agglomerations.xlsx'
df_poblacion_raw = cargar_excel_seguro(path_poblacion, sheet_name='City Population Size ')

# Reconstruir columnas con la fila 1 que contiene los años
header_row = df_poblacion_raw.iloc[1].tolist()
df_poblacion = df_poblacion_raw.iloc[2:].copy()
df_poblacion.columns = ['Country', 'City', 'Unit'] + [str(int(float(y))) for y in header_row[3:]]

print(f'Dimensiones de Población: {df_poblacion.shape[0]} ciudades x {df_poblacion.shape[1]} columnas')
df_poblacion.head(3)

Dimensiones de Población: 1867 ciudades x 11 columnas


,Country,City,Unit,2000,2005,2010,2015,2020,2025,2030,2035
2,Afghanistan,Herat,Thousand,233.991,275.678,358.691,466.703,605.575,752.910,897.041,1057.573
3,Afghanistan,Kabul,Thousand,2401.109,2905.178,3289.005,3723.543,4221.532,4877.024,5737.138,6760.500
4,Afghanistan,Kandahar,Thousand,297.456,336.746,383.498,436.741,498.002,577.128,679.278,800.461


## 6. Acceso al Transporte Público

Porcentaje de la población con acceso conveniente a transporte público (ODS 11.2.1).

In [31]:
path_transporte = 'data/raw/urban-indicator-database-features/urban-transport/public_transport_access.xlsx'
df_transporte = cargar_excel_seguro(path_transporte, sheet_name='Sheet1')

df_transporte = df_transporte.rename(columns={'Country or Territory Name': 'Country', 'City Name': 'City'})
print(f'Dimensiones de Transporte: {df_transporte.shape[0]} ciudades x {df_transporte.shape[1]} columnas')
df_transporte.head(3)

Dimensiones de Transporte: 2353 ciudades x 14 columnas


,SDG Goal,SDG Target,SDG Indicator,Country or Territory Code,Country,SDG Region,SDG Sub-Region,City Code,City,Share of urban population with convenient access to public transport (%),Data Units,Data Reference Year,Data Source,FootNote
0,11,11.2,11.2.1,4,Afghanistan,Central Asia and Southern Asia,Southern Asia,AF_CHARIKAR,Chārīkār,77.513852,PERCENT,2020,UN-Habitat Urban Indicators Database,Only public transport stops which are mapped a...
1,11,11.2,11.2.1,4,Afghanistan,Central Asia and Southern Asia,Southern Asia,AF_FARAH,Farāh,69.499376,PERCENT,2020,UN-Habitat Urban Indicators Database,Only public transport stops which are mapped a...
2,11,11.2,11.2.1,4,Afghanistan,Central Asia and Southern Asia,Southern Asia,AF_HERAT,Herat,50.358416,PERCENT,2020,UN-Habitat Urban Indicators Database,Only public transport stops which are mapped a...
